In [46]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)

c:\Users\Deepti Bhardwaj\anaconda3\python.exe
1.26.4


In [47]:
import sys
print(sys.executable)

c:\Users\Deepti Bhardwaj\anaconda3\python.exe


In [48]:
import pandas as pd
import numpy as np
df = pd.read_csv(
"feature_engineered.csv"
)
df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter,Customer_Activity_Score,Revenue_Per_Purchase,Inactivity_Risk,Customer_Friction_Index,Discount_Dependency,Loyalty_Score
0,43.0,Male,France,Marseille,2.9,0.304348,0.353887,0.216450,50.6,3.0,...,953.33,2278.0,0,Q1,0.281183,95.333000,26.066667,14.06,417.600,3.951304
1,36.0,Male,UK,Manchester,1.6,0.326087,0.558981,0.402597,37.7,1.0,...,1067.47,3028.0,0,Q4,0.426164,52.071707,53.540984,10.77,1130.220,6.587826
2,45.0,Female,Canada,Vancouver,2.9,0.217391,0.319035,0.025974,70.9,1.0,...,1289.75,2317.0,0,Q4,0.140913,127.698020,9.035714,11.09,111.384,3.955217
3,56.0,Female,USA,New York,2.6,0.217391,0.501340,0.597403,41.7,9.0,...,2340.92,2674.0,0,Q1,0.453683,146.307500,38.607143,6.17,661.500,5.605217
4,35.0,Male,India,Delhi,3.1,0.630435,0.675603,0.320346,19.1,9.0,...,3041.29,5354.0,0,Q4,0.570781,90.784776,44.773333,2.91,819.000,11.179130


In [173]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

seg_cols = [
    "Customer_Activity_Score",
    "Loyalty_Score",
    "Lifetime_Value",
    "Total_Purchases",
    "Customer_Friction_Index",
    "Inactivity_Risk",
    "Days_Since_Last_Purchase",
    "Customer_Service_Calls"
]

scaler = StandardScaler()

X = scaler.fit_transform(df[seg_cols])
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)

df["Cluster"] = cluster_labels

In [174]:
df["Cluster"].value_counts()

Cluster
2    18735
3    16327
0     9286
1     5652
Name: count, dtype: int64

In [175]:
X = df.drop("Churned",axis=1)

y = df["Churned"]

In [176]:
X = pd.get_dummies(
X,
drop_first=True
)

In [177]:
print(df["Cluster"].nunique())

4


In [178]:
print(df["Cluster"].value_counts())

Cluster
2    18735
3    16327
0     9286
1     5652
Name: count, dtype: int64


In [179]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [180]:
X_train.isnull().sum().sum()

0

In [181]:
import numpy as np

np.isinf(X_train).sum().sum()

2

In [182]:
# Replace infinite values in the split data
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

In [183]:
# Fill missing values using training-set medians to avoid leakage
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

In [184]:
np.isinf(X_train).sum().sum()

0

In [185]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
n_estimators=400,
max_depth=8,
class_weight="balanced",
random_state=42
)

rf.fit(
X_train,
y_train
)

RandomForestClassifier(class_weight='balanced', max_depth=8, n_estimators=400,
                       random_state=42)

In [186]:
pred = rf.predict(X_test)

prob = rf.predict_proba(
X_test
)[:,1]

In [187]:
from sklearn.metrics import (
classification_report,
confusion_matrix,
roc_auc_score
)

print(
classification_report(
y_test,
pred
)
)

print(
roc_auc_score(
y_test,
prob
)
)

              precision    recall  f1-score   support

           0       0.94      0.89      0.91      7110
           1       0.76      0.85      0.81      2890

    accuracy                           0.88     10000
   macro avg       0.85      0.87      0.86     10000
weighted avg       0.89      0.88      0.88     10000

0.9135289737637422


In [188]:
importance = pd.Series(
rf.feature_importances_,
index=X_train.columns
)

importance = importance.sort_values(
ascending=False
)

importance.head(15)

Customer_Service_Calls      0.127836
Lifetime_Value              0.117129
Customer_Friction_Index     0.105837
Cart_Abandonment_Rate       0.087160
Customer_Activity_Score     0.080680
Age                         0.061190
Total_Purchases             0.039948
Revenue_Per_Purchase        0.039067
Average_Order_Value         0.032150
Inactivity_Risk             0.030130
Discount_Usage_Rate         0.028783
Email_Open_Rate             0.026701
Discount_Dependency         0.025578
Pages_Per_Session           0.024024
Days_Since_Last_Purchase    0.022254
dtype: float64

Recommendation 1
Target high-friction customers

Trigger:
Customer_Friction_Index high

Action:
Priority support 
Checkout improvements

Recommendation 2
Protect high lifetime value customers

Trigger:

Lifetime_Value high
AND
Churn probability high

Action:

Premium retention campaigns

Recommendation 3
Reduce cart abandonment

Action:

Checkout optimization experiments

In [189]:
import joblib

joblib.dump(
    rf,
    "../models/churn_rf.pkl"
)

['../models/churn_rf.pkl']

In [190]:
df.to_csv(
"../data/final_customer_data.csv",
index=False
)

In [191]:
import joblib

joblib.dump(
X_train.columns.tolist(),
"../models/feature_columns.pkl"
)

['../models/feature_columns.pkl']

In [192]:
reference_customer = X.median(
    numeric_only=True
)
reference_customer = (
    reference_customer
    .reindex(X.columns)
    .fillna(0)
)

joblib.dump(
reference_customer,
"../models/reference_customer.pkl"
)

['../models/reference_customer.pkl']

In [193]:
import pandas as pd
import numpy as np
from sklearn.utils.validation import check_is_fitted

# Quick diagnostics to understand model / data state
print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('Number of features (train):', X_train.shape[1])
print('y_train class balance:\n', y_train.value_counts())
print('NaNs in X_train / X_test:', X_train.isnull().sum().sum(), X_test.isnull().sum().sum())
print('Infs in X_train / X_test:', np.isinf(X_train).sum().sum(), np.isinf(X_test).sum().sum())

try:
    check_is_fitted(rf)
    print('rf is fitted')
except Exception as e:
    print('rf not fitted:', e)

fi_len = len(getattr(rf, 'feature_importances_', []))
col_len = X_train.shape[1]
print('feature_importances_ length =', fi_len, 'X_train n_features =', col_len)
if fi_len == col_len and fi_len > 0:
    importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    display(importance.head(20))
else:
    print('Length mismatch; cannot build importance Series. Use X_train.columns or retrain the model.')

X_train shape: (40000, 78)
X_test shape: (10000, 78)
Number of features (train): 78
y_train class balance:
 Churned
0    28440
1    11560
Name: count, dtype: int64
NaNs in X_train / X_test: 0 0
Infs in X_train / X_test: 0 0
rf is fitted
feature_importances_ length = 78 X_train n_features = 78


Customer_Service_Calls      0.127836
Lifetime_Value              0.117129
Customer_Friction_Index     0.105837
Cart_Abandonment_Rate       0.087160
Customer_Activity_Score     0.080680
Age                         0.061190
Total_Purchases             0.039948
Revenue_Per_Purchase        0.039067
Average_Order_Value         0.032150
Inactivity_Risk             0.030130
Discount_Usage_Rate         0.028783
Email_Open_Rate             0.026701
Discount_Dependency         0.025578
Pages_Per_Session           0.024024
Days_Since_Last_Purchase    0.022254
Session_Duration_Avg        0.020659
Cluster                     0.020259
Mobile_App_Usage            0.019339
Loyalty_Score               0.018649
Wishlist_Items              0.018288
dtype: float64

In [194]:
# Build a 'best' reference row using training medians, then overwrite desirable values
best = X_train.median().to_frame().T

best["Lifetime_Value"] = 8000
best["Cart_Abandonment_Rate"] = 0
best["Customer_Service_Calls"] = 0
best["Customer_Friction_Index"] = 0
best["Customer_Activity_Score"] = 1
best["Inactivity_Risk"] = 0
best["Total_Purchases"] = 50
best["Days_Since_Last_Purchase"] = 1
best["Loyalty_Score"] = 10
best["Email_Open_Rate"] = 1
best["Login_Frequency"] = 20

# Ensure correct feature order and add any missing dummy columns as 0
best = best.reindex(columns=X_train.columns, fill_value=0)

In [195]:
# Build a 'worst' reference row using training medians as base
worst = X_train.median().to_frame().T

worst["Lifetime_Value"] = 0
worst["Cart_Abandonment_Rate"] = 100
worst["Customer_Service_Calls"] = 20
worst["Customer_Friction_Index"] = 20
worst["Customer_Activity_Score"] = 0
worst["Inactivity_Risk"] = 1
worst["Total_Purchases"] = 0
worst["Days_Since_Last_Purchase"] = 365
worst["Loyalty_Score"] = 0
worst["Email_Open_Rate"] = 0
worst["Login_Frequency"] = 0

# Ensure correct feature order
worst = worst.reindex(columns=X_train.columns, fill_value=0)

In [196]:
# Ensure best/worst use training columns
best = best.reindex(columns=X_train.columns, fill_value=0)
worst = worst.reindex(columns=X_train.columns, fill_value=0)

print('BEST:', rf.predict_proba(best)[0][1])
print('WORST:', rf.predict_proba(worst)[0][1])

BEST: 0.24900986876338732
WORST: 0.7920424991653487


In [197]:
pred = rf.predict(X_test)
prob = rf.predict_proba(X_test)[:,1]

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test,pred))
print("AUC:",roc_auc_score(y_test,prob))

              precision    recall  f1-score   support

           0       0.94      0.89      0.91      7110
           1       0.76      0.85      0.81      2890

    accuracy                           0.88     10000
   macro avg       0.85      0.87      0.86     10000
weighted avg       0.89      0.88      0.88     10000

AUC: 0.9135289737637422


In [198]:
print(X_train.shape)

(40000, 78)


In [199]:
print(rf.get_params()["class_weight"])

balanced


In [200]:
print(reference_customer.shape)

(78,)


In [201]:
importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importance.head(15)

Customer_Service_Calls      0.127836
Lifetime_Value              0.117129
Customer_Friction_Index     0.105837
Cart_Abandonment_Rate       0.087160
Customer_Activity_Score     0.080680
Age                         0.061190
Total_Purchases             0.039948
Revenue_Per_Purchase        0.039067
Average_Order_Value         0.032150
Inactivity_Risk             0.030130
Discount_Usage_Rate         0.028783
Email_Open_Rate             0.026701
Discount_Dependency         0.025578
Pages_Per_Session           0.024024
Days_Since_Last_Purchase    0.022254
dtype: float64

In [202]:
df.groupby("Cluster")[
[
"Customer_Activity_Score",
"Loyalty_Score",
"Lifetime_Value",
"Customer_Friction_Index",
"Total_Purchases"
]
].mean()

,Customer_Activity_Score,Loyalty_Score,Lifetime_Value,Customer_Friction_Index,Total_Purchases
Cluster,,,,,
0,0.480036,8.242623,2572.765708,7.319268,22.772216
1,0.264987,4.761953,1271.443206,11.829248,11.688889
2,0.305118,5.252015,1435.928235,10.273669,13.238046
3,0.193288,3.570308,860.678615,14.826609,7.964452


In [203]:
df.groupby("Cluster")["Churned"].mean()

Cluster
0    0.202455
1    0.450460
2    0.192581
3    0.392969
Name: Churned, dtype: float64

In [204]:
df.to_csv(
    "../data/final_customer_data.csv",
    index=False
)